In [13]:
%pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [1]:
import yfinance as yf
import pandas as pd
from pathlib import Path

aapl_5y = yf.download(
    "AAPL",
    start="2021-08-31",
    end="2026-09-01",
    interval="1d",
    auto_adjust=True,
    progress=False
)

print("Rows:", len(aapl_5y))
print("First date:", aapl_5y.index.min())
print("Last date:", aapl_5y.index.max())

display(aapl_5y.head())

Rows: 1254
First date: 2021-08-31 00:00:00
Last date: 2026-08-28 00:00:00


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2021-08-31,148.090607,149.036718,147.563898,148.900167,86453100
2021-09-01,148.753860,151.163029,148.588049,149.065986,80313700
2021-09-02,149.865768,150.909423,148.646555,150.080351,71115500
2021-09-03,150.499756,150.821630,149.319550,149.973047,57808700
2021-09-07,152.830902,153.386856,150.587546,151.153263,82278300


In [3]:
import time

tickers = ["AAPL", "MSFT", "NVDA", "QQQ"]

raw_5y_dir = Path("data/raw_5y_yahoo")
raw_5y_dir.mkdir(parents=True, exist_ok=True)

all_5y_data = []

for ticker in tickers:
    print(f"Downloading {ticker}...")

    df = yf.download(
        ticker,
        start="2021-08-31",
        end="2026-09-01",
        interval="1d",
        auto_adjust=True,
        progress=False
    )

    # Flatten yfinance multi-level columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df.insert(0, "Ticker", ticker)

    # Keep and reorder required daily OHLCV columns
    df = df[
        ["Ticker", "Date", "Open", "High", "Low", "Close", "Volume"]
    ]

    file_path = raw_5y_dir / f"{ticker}_daily_5y.csv"
    df.to_csv(file_path, index=False)

    print(f"{ticker}: {len(df)} rows saved to {file_path}")

    all_5y_data.append(df)
    time.sleep(1)

combined_5y_df = pd.concat(all_5y_data, ignore_index=True)

combined_file = raw_5y_dir / "all_tickers_daily_5y.csv"
combined_5y_df.to_csv(combined_file, index=False)

print("\nDownload complete.")
print("Combined rows:", len(combined_5y_df))
print("Combined file:", combined_file)

AAPL: 1254 rows saved to data\raw_5y_yahoo\AAPL_daily_5y.csv
MSFT: 1254 rows saved to data\raw_5y_yahoo\MSFT_daily_5y.csv
NVDA: 1254 rows saved to data\raw_5y_yahoo\NVDA_daily_5y.csv
QQQ: 1254 rows saved to data\raw_5y_yahoo\QQQ_daily_5y.csv

Download complete.
Combined rows: 5016
Combined file: data\raw_5y_yahoo\all_tickers_daily_5y.csv


In [5]:
print("=== Basic summary ===")

summary_5y = combined_5y_df.groupby("Ticker").agg(
    Rows=("Date", "size"),
    Start_Date=("Date", "min"),
    End_Date=("Date", "max"),
    Missing_Close=("Close", lambda x: x.isna().sum())
)

display(summary_5y)

print("\n=== Duplicate ticker-date rows ===")

duplicate_count_5y = combined_5y_df.duplicated(
    subset=["Ticker", "Date"]
).sum()

print(duplicate_count_5y)

print("\n=== Invalid OHLC rows ===")

invalid_ohlc_5y = combined_5y_df[
    (combined_5y_df["High"] <
     combined_5y_df[["Open", "Close", "Low"]].max(axis=1))
    |
    (combined_5y_df["Low"] >
     combined_5y_df[["Open", "Close", "High"]].min(axis=1))
]

print(len(invalid_ohlc_5y))

print("\n=== Missing values in all columns ===")

display(combined_5y_df.isna().sum())

=== Basic summary ===


,Rows,Start_Date,End_Date,Missing_Close
Ticker,,,,
AAPL,1254,2021-08-31,2026-08-28,1
MSFT,1254,2021-08-31,2026-08-28,1
NVDA,1254,2021-08-31,2026-08-28,1
QQQ,1254,2021-08-31,2026-08-28,1



=== Duplicate ticker-date rows ===
0

=== Invalid OHLC rows ===
0

=== Missing values in all columns ===


Price
Ticker    0
Date      0
Open      4
High      4
Low       4
Close     4
Volume    0
dtype: int64

In [7]:
missing_rows_5y = combined_5y_df[
    combined_5y_df[
        ["Open", "High", "Low", "Close", "Volume"]
    ].isna().any(axis=1)
]

print("Rows containing missing OHLCV values:")
display(missing_rows_5y)

Rows containing missing OHLCV values:


Price,Ticker,Date,Open,High,Low,Close,Volume
1253,AAPL,2026-08-28,NaN,NaN,NaN,NaN,38623623
2507,MSFT,2026-08-28,NaN,NaN,NaN,NaN,29189683
3761,NVDA,2026-08-28,NaN,NaN,NaN,NaN,194792961
5015,QQQ,2026-08-28,NaN,NaN,NaN,NaN,34067577


In [9]:
# Preserve the original downloaded data
cleaned_5y_df = combined_5y_df.dropna(
    subset=["Open", "High", "Low", "Close", "Volume"]
).copy()

cleaned_5y_df = cleaned_5y_df.sort_values(
    ["Date", "Ticker"]
).reset_index(drop=True)

print("Original rows:", len(combined_5y_df))
print("Cleaned rows:", len(cleaned_5y_df))
print("Removed rows:", len(combined_5y_df) - len(cleaned_5y_df))

display(
    cleaned_5y_df.groupby("Ticker").agg(
        Rows=("Date", "size"),
        Start_Date=("Date", "min"),
        End_Date=("Date", "max")
    )
)

Original rows: 5016
Cleaned rows: 5012
Removed rows: 4


,Rows,Start_Date,End_Date
Ticker,,,
AAPL,1253,2021-08-31,2026-08-27
MSFT,1253,2021-08-31,2026-08-27
NVDA,1253,2021-08-31,2026-08-27
QQQ,1253,2021-08-31,2026-08-27


In [11]:
print("=== Number of tickers available on each date ===")

date_counts_5y = cleaned_5y_df.groupby("Date")["Ticker"].nunique()
display(date_counts_5y.value_counts().sort_index())

incomplete_dates_5y = date_counts_5y[date_counts_5y != 4]

print("\nDates without all four tickers:", len(incomplete_dates_5y))

if len(incomplete_dates_5y) > 0:
    display(incomplete_dates_5y)
else:
    print("All dates contain AAPL, MSFT, NVDA and QQQ.")

=== Number of tickers available on each date ===


Ticker
4    1253
Name: count, dtype: int64


Dates without all four tickers: 0
All dates contain AAPL, MSFT, NVDA and QQQ.


In [13]:
processed_5y_dir = Path("data/processed_5y_yahoo")
processed_5y_dir.mkdir(parents=True, exist_ok=True)

aligned_5y_file = processed_5y_dir / "aligned_daily_ohlcv_5y.csv"

cleaned_5y_df.to_csv(
    aligned_5y_file,
    index=False
)

print("Saved:", aligned_5y_file)
print("Rows:", len(cleaned_5y_df))
print("Columns:", len(cleaned_5y_df.columns))

display(cleaned_5y_df.head(8))

Saved: data\processed_5y_yahoo\aligned_daily_ohlcv_5y.csv
Rows: 5012
Columns: 7


Price,Ticker,Date,Open,High,Low,Close,Volume
0,AAPL,2021-08-31,148.900167,149.036718,147.563898,148.090607,86453100
1,MSFT,2021-08-31,292.160127,292.236892,289.357711,289.722412,26285300
2,NVDA,2021-08-31,22.620757,22.620757,22.047639,22.311771,259850000
3,QQQ,2021-08-31,369.280898,369.348810,367.194323,368.737427,29628200
4,AAPL,2021-09-01,149.065971,151.163013,148.588034,148.753845,80313700
5,MSFT,2021-09-01,290.672594,292.899168,289.348165,289.674469,18983800
6,NVDA,2021-09-01,22.411449,22.622755,22.283867,22.367592,201767000
7,QQQ,2021-09-01,369.795232,371.415932,369.144991,369.348785,28138300


In [15]:
cleaned_5y_df.columns.name = None
cleaned_5y_df.to_csv(aligned_5y_file, index=False)

display(cleaned_5y_df.head())

,Ticker,Date,Open,High,Low,Close,Volume
0,AAPL,2021-08-31,148.900167,149.036718,147.563898,148.090607,86453100
1,MSFT,2021-08-31,292.160127,292.236892,289.357711,289.722412,26285300
2,NVDA,2021-08-31,22.620757,22.620757,22.047639,22.311771,259850000
3,QQQ,2021-08-31,369.280898,369.348810,367.194323,368.737427,29628200
4,AAPL,2021-09-01,149.065971,151.163013,148.588034,148.753845,80313700


In [3]:
# Final audit: reload files from disk

import pandas as pd
from pathlib import Path

aligned_5y_file = Path(
    "data/processed_5y_yahoo/aligned_daily_ohlcv_5y.csv"
)
raw_files = {
    "AAPL": Path("data/raw_5y_yahoo/AAPL_daily_5y.csv"),
    "MSFT": Path("data/raw_5y_yahoo/MSFT_daily_5y.csv"),
    "NVDA": Path("data/raw_5y_yahoo/NVDA_daily_5y.csv"),
    "QQQ": Path("data/raw_5y_yahoo/QQQ_daily_5y.csv")
}

print("=== RAW FILE AUDIT ===")

for ticker, file_path in raw_files.items():
    raw_df = pd.read_csv(file_path)

    missing_ohlc = raw_df[
        ["Open", "High", "Low", "Close"]
    ].isna().any(axis=1).sum()

    duplicates = raw_df.duplicated(
        subset=["Ticker", "Date"]
    ).sum()

    print(
        f"{ticker}: "
        f"rows={len(raw_df)}, "
        f"missing_OHLC_rows={missing_ohlc}, "
        f"duplicates={duplicates}"
    )

print("\n=== PROCESSED FILE AUDIT ===")

processed_check = pd.read_csv(aligned_5y_file)

print("Rows:", len(processed_check))
print("Columns:", list(processed_check.columns))
print("Start date:", processed_check["Date"].min())
print("End date:", processed_check["Date"].max())
print("Missing values:", processed_check.isna().sum().sum())
print(
    "Duplicate ticker-date rows:",
    processed_check.duplicated(["Ticker", "Date"]).sum()
)

print("\nRows per ticker:")
display(processed_check["Ticker"].value_counts().sort_index())

=== RAW FILE AUDIT ===
AAPL: rows=1254, missing_OHLC_rows=1, duplicates=0
MSFT: rows=1254, missing_OHLC_rows=1, duplicates=0
NVDA: rows=1254, missing_OHLC_rows=1, duplicates=0
QQQ: rows=1254, missing_OHLC_rows=1, duplicates=0

=== PROCESSED FILE AUDIT ===
Rows: 5012
Columns: ['Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume']
Start date: 2021-08-31
End date: 2026-08-27
Missing values: 0
Duplicate ticker-date rows: 0

Rows per ticker:


Ticker
AAPL    1253
MSFT    1253
NVDA    1253
QQQ     1253
Name: count, dtype: int64

In [5]:
data_note = """# Data Documentation

## Overview

This dataset contains daily market data for AAPL, MSFT, NVDA, and QQQ. It is intended for subsequent feature construction, target creation, and modelling experiments.

## Data Source

- Source: Yahoo Finance
- Download tool: Python `yfinance`
- Download interval: Daily (`1d`)
- Requested period: 2021-08-31 to 2026-09-01
- Price adjustment: `auto_adjust=True`
- No API key is required.

Because `auto_adjust=True` was used, the Open, High, Low, and Close columns contain adjusted prices.

## Columns

The raw and processed datasets use the following columns:

- `Ticker`: Instrument symbol
- `Date`: Trading date in `YYYY-MM-DD` format
- `Open`: Adjusted opening price
- `High`: Adjusted highest price
- `Low`: Adjusted lowest price
- `Close`: Adjusted closing price
- `Volume`: Daily trading volume

## Raw Data

The four raw files are:

- `AAPL_daily_5y.csv`
- `MSFT_daily_5y.csv`
- `NVDA_daily_5y.csv`
- `QQQ_daily_5y.csv`

The raw files have been preserved without modification. Each raw file contains 1,254 rows covering 2021-08-31 to 2026-08-28.

## Data Quality Checks

The following checks were performed:

- Missing-value check
- Duplicate ticker-date check
- OHLC consistency check
- Chronological sorting
- Common trading-date alignment across all four instruments
- Column consistency check

No duplicate ticker-date records or invalid OHLC relationships were found.

## Removed Records

Four incomplete records dated 2026-08-28 were removed from the processed dataset:

- AAPL — 2026-08-28
- MSFT — 2026-08-28
- NVDA — 2026-08-28
- QQQ — 2026-08-28

These records contained Volume values, but Open, High, Low, and Close were all missing. The values were not imputed because doing so would create artificial price data.

The original records remain unchanged in the raw files.

## Final Processed Dataset

File:

`data/processed_5y_yahoo/aligned_daily_ohlcv_5y.csv`

Final dataset details:

- Date range: 2021-08-31 to 2026-08-27
- Rows per instrument: 1,253
- Total rows: 5,012
- Missing values: 0
- Duplicate ticker-date rows: 0
- All four instruments use the same 1,253 common trading dates
- Data is sorted chronologically by Date and Ticker

This processed dataset should be used as the shared input for the next feature and modelling stage.

## Reproducibility

The accompanying Jupyter Notebook contains the download, validation, cleaning, alignment, and export process. No API key or credential is stored in the notebook.
"""

readme_file = Path("DATA_README.md")
readme_file.write_text(data_note, encoding="utf-8")

print("Created:", readme_file)
print("Characters:", len(data_note))

Created: DATA_README.md
Characters: 2478


In [1]:
import pandas as pd
from pathlib import Path

# Existing five-year cleaned dataset
five_year_file = Path(
    "data/processed_5y_yahoo/aligned_daily_ohlcv_5y.csv"
)

print("Current working directory:", Path.cwd())
print("File exists:", five_year_file.exists())

if not five_year_file.exists():
    raise FileNotFoundError(
        f"Cannot find: {five_year_file}\n"
        "Please check the notebook working directory and file path."
    )

df_5y = pd.read_csv(five_year_file)

# Standardise the date type again after loading the CSV
df_5y["Date"] = pd.to_datetime(
    df_5y["Date"],
    format="%Y-%m-%d",
    errors="raise"
)

# Sort without modifying the original CSV file
df_5y = (
    df_5y
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

print("\n=== FIVE-YEAR CLEANED DATASET ===")
print("Rows:", len(df_5y))
print("Columns:", df_5y.columns.tolist())
print("Start date:", df_5y["Date"].min().date())
print("End date:", df_5y["Date"].max().date())
print("Tickers:", sorted(df_5y["Ticker"].unique()))

print("\nRows per ticker:")
display(df_5y["Ticker"].value_counts().sort_index())

display(df_5y.head(8))

Current working directory: C:\Users\11246\Desktop\ict项目
File exists: True

=== FIVE-YEAR CLEANED DATASET ===
Rows: 5012
Columns: ['Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume']
Start date: 2021-08-31
End date: 2026-08-27
Tickers: ['AAPL', 'MSFT', 'NVDA', 'QQQ']

Rows per ticker:


Ticker
AAPL    1253
MSFT    1253
NVDA    1253
QQQ     1253
Name: count, dtype: int64

,Ticker,Date,Open,High,Low,Close,Volume
0,AAPL,2021-08-31,148.900167,149.036718,147.563898,148.090607,86453100
1,MSFT,2021-08-31,292.160127,292.236892,289.357711,289.722412,26285300
2,NVDA,2021-08-31,22.620757,22.620757,22.047639,22.311771,259850000
3,QQQ,2021-08-31,369.280898,369.348810,367.194323,368.737427,29628200
4,AAPL,2021-09-01,149.065971,151.163013,148.588034,148.753845,80313700
5,MSFT,2021-09-01,290.672594,292.899168,289.348165,289.674469,18983800
6,NVDA,2021-09-01,22.411449,22.622755,22.283867,22.367592,201767000
7,QQQ,2021-09-01,369.795232,371.415932,369.144991,369.348785,28138300


In [3]:
# Extract the full 2025 dataset
df_2025 = df_5y[
    df_5y["Date"].dt.year == 2025
].copy()

df_2025 = (
    df_2025
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

print("=== 2025 DATASET CHECK ===")
print("Rows:", len(df_2025))
print("Start date:", df_2025["Date"].min().date())
print("End date:", df_2025["Date"].max().date())
print("Missing values:", int(df_2025.isna().sum().sum()))
print(
    "Duplicate ticker-date rows:",
    int(df_2025.duplicated(["Ticker", "Date"]).sum())
)

print("\nRows per ticker:")
display(
    df_2025["Ticker"]
    .value_counts()
    .sort_index()
)

# Check how many instruments exist on each trading date
tickers_per_date = (
    df_2025
    .groupby("Date")["Ticker"]
    .nunique()
)

incomplete_dates = tickers_per_date[
    tickers_per_date != 4
]

print("\nNumber of common trading dates:", df_2025["Date"].nunique())
print("Dates without all four instruments:", len(incomplete_dates))

if len(incomplete_dates) == 0:
    print("All 2025 dates contain AAPL, MSFT, NVDA and QQQ.")
else:
    display(incomplete_dates)

display(df_2025.head(8))
display(df_2025.tail(8))

=== 2025 DATASET CHECK ===
Rows: 1000
Start date: 2025-01-02
End date: 2025-12-31
Missing values: 0
Duplicate ticker-date rows: 0

Rows per ticker:


Ticker
AAPL    250
MSFT    250
NVDA    250
QQQ     250
Name: count, dtype: int64


Number of common trading dates: 250
Dates without all four instruments: 0
All 2025 dates contain AAPL, MSFT, NVDA and QQQ.


,Ticker,Date,Open,High,Low,Close,Volume
0,AAPL,2025-01-02,247.136526,247.305315,240.077766,242.093140,55740700
1,MSFT,2025-01-02,419.750981,420.283656,409.216031,412.895355,16896500
2,NVDA,2025-01-02,135.797207,138.672917,134.429254,138.103760,198247200
3,QQQ,2025-01-02,510.499434,512.822169,501.972916,506.459534,36389800
4,AAPL,2025-01-03,241.606659,242.420743,240.147249,241.606659,40244100
5,MSFT,2025-01-03,415.361395,418.271344,413.842331,417.600586,16662900
6,NVDA,2025-01-03,139.801207,144.683914,139.521625,144.254562,229322500
7,QQQ,2025-01-03,509.556531,515.810025,508.742644,514.747925,29059500


,Ticker,Date,Open,High,Low,Close,Volume
992,AAPL,2025-12-30,272.069428,273.335969,271.540868,272.338684,22139600
993,MSFT,2025-12-30,482.866614,486.592973,482.439332,484.406860,13944500
994,NVDA,2025-12-30,188.010698,188.759785,186.702281,187.311539,97687300
995,QQQ,2025-12-30,618.379832,620.714286,617.761237,617.970764,31226800
996,AAPL,2025-12-31,272.318764,272.937076,271.012322,271.122009,27293600
997,MSFT,2025-12-31,484.764565,485.062692,480.253177,480.571167,15601600
998,NVDA,2025-12-31,189.339064,190.327848,186.262814,186.272797,120100500
999,QQQ,2025-12-31,618.190301,618.499569,612.603457,612.862854,40746500


In [5]:
# Add calendar-quarter labels
df_2025["Quarter"] = (
    "Q" + df_2025["Date"].dt.quarter.astype(str)
)

# Q1-Q2 = training
# Q3 = validation
# Q4 = testing
split_mapping = {
    "Q1": "train",
    "Q2": "train",
    "Q3": "validation",
    "Q4": "test"
}

df_2025["Split"] = df_2025["Quarter"].map(split_mapping)

# Keep the dataset in chronological order
df_2025 = (
    df_2025
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

print("=== QUARTERLY SPLIT SUMMARY ===")

summary = (
    df_2025
    .groupby(["Quarter", "Split"])
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Trading_Dates=("Date", "nunique"),
        Total_Rows=("Date", "size")
    )
    .reset_index()
)

display(summary)

print("\nRows by split:")
display(df_2025["Split"].value_counts())

print("\nMissing split labels:", df_2025["Split"].isna().sum())

display(
    df_2025[
        ["Ticker", "Date", "Quarter", "Split",
         "Open", "High", "Low", "Close", "Volume"]
    ].head(8)
)

=== QUARTERLY SPLIT SUMMARY ===


,Quarter,Split,Start_Date,End_Date,Trading_Dates,Total_Rows
0,Q1,train,2025-01-02,2025-03-31,60,240
1,Q2,train,2025-04-01,2025-06-30,62,248
2,Q3,validation,2025-07-01,2025-09-30,64,256
3,Q4,test,2025-10-01,2025-12-31,64,256



Rows by split:


Split
train         488
validation    256
test          256
Name: count, dtype: int64


Missing split labels: 0


,Ticker,Date,Quarter,Split,Open,High,Low,Close,Volume
0,AAPL,2025-01-02,Q1,train,247.136526,247.305315,240.077766,242.093140,55740700
1,MSFT,2025-01-02,Q1,train,419.750981,420.283656,409.216031,412.895355,16896500
2,NVDA,2025-01-02,Q1,train,135.797207,138.672917,134.429254,138.103760,198247200
3,QQQ,2025-01-02,Q1,train,510.499434,512.822169,501.972916,506.459534,36389800
4,AAPL,2025-01-03,Q1,train,241.606659,242.420743,240.147249,241.606659,40244100
5,MSFT,2025-01-03,Q1,train,415.361395,418.271344,413.842331,417.600586,16662900
6,NVDA,2025-01-03,Q1,train,139.801207,144.683914,139.521625,144.254562,229322500
7,QQQ,2025-01-03,Q1,train,509.556531,515.810025,508.742644,514.747925,29059500


In [7]:
# Final audit before export

required_columns = [
    "Ticker", "Date", "Quarter", "Split",
    "Open", "High", "Low", "Close", "Volume"
]

# Check required columns
missing_columns = [
    column for column in required_columns
    if column not in df_2025.columns
]

# Check invalid OHLC relationships
invalid_ohlc = df_2025[
    (df_2025["High"] < df_2025["Low"]) |
    (df_2025["Open"] > df_2025["High"]) |
    (df_2025["Open"] < df_2025["Low"]) |
    (df_2025["Close"] > df_2025["High"]) |
    (df_2025["Close"] < df_2025["Low"])
]

# Check ticker coverage on each date
tickers_per_date = (
    df_2025
    .groupby("Date")["Ticker"]
    .nunique()
)

incomplete_dates = tickers_per_date[
    tickers_per_date != 4
]

# Obtain date boundaries
train_end = df_2025.loc[
    df_2025["Split"] == "train", "Date"
].max()

validation_start = df_2025.loc[
    df_2025["Split"] == "validation", "Date"
].min()

validation_end = df_2025.loc[
    df_2025["Split"] == "validation", "Date"
].max()

test_start = df_2025.loc[
    df_2025["Split"] == "test", "Date"
].min()

print("=== FINAL 2025 DATA AUDIT ===")
print("Rows:", len(df_2025))
print("Columns:", len(df_2025.columns))
print("Missing required columns:", missing_columns)
print("Missing values:", int(df_2025.isna().sum().sum()))
print(
    "Duplicate ticker-date rows:",
    int(df_2025.duplicated(["Ticker", "Date"]).sum())
)
print("Invalid OHLC rows:", len(invalid_ohlc))
print("Negative volume rows:", int((df_2025["Volume"] < 0).sum()))
print("Dates without all four tickers:", len(incomplete_dates))

print("\n=== TIME BOUNDARIES ===")
print("Train end:", train_end.date())
print("Validation start:", validation_start.date())
print("Validation end:", validation_end.date())
print("Test start:", test_start.date())

chronological_order_valid = (
    train_end < validation_start < validation_end < test_start
)

print(
    "Chronological split order valid:",
    chronological_order_valid
)

=== FINAL 2025 DATA AUDIT ===
Rows: 1000
Columns: 9
Missing required columns: []
Missing values: 0
Duplicate ticker-date rows: 0
Invalid OHLC rows: 0
Negative volume rows: 0
Dates without all four tickers: 0

=== TIME BOUNDARIES ===
Train end: 2025-06-30
Validation start: 2025-07-01
Validation end: 2025-09-30
Test start: 2025-10-01
Chronological split order valid: True


In [9]:
# Arrange the final column order
final_2025_df = df_2025[
    [
        "Ticker",
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Quarter",
        "Split"
    ]
].copy()

# Create a separate folder for the one-year dataset
one_year_processed_dir = Path(
    "data/processed_1y_yahoo"
)

one_year_processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    one_year_processed_dir /
    "aligned_daily_ohlcv_2025_split.csv"
)

# Export without modifying the five-year dataset
final_2025_df.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved:", output_file)
print("Rows:", len(final_2025_df))
print("Columns:", len(final_2025_df.columns))
print("File exists:", output_file.exists())

Saved: data\processed_1y_yahoo\aligned_daily_ohlcv_2025_split.csv
Rows: 1000
Columns: 9
File exists: True


In [11]:
# Reload the exported CSV from disk

exported_check = pd.read_csv(
    output_file,
    parse_dates=["Date"]
)

print("=== EXPORTED FILE CHECK ===")
print("File:", output_file)
print("Rows:", len(exported_check))
print("Columns:", exported_check.columns.tolist())
print("Start date:", exported_check["Date"].min().date())
print("End date:", exported_check["Date"].max().date())
print("Tickers:", sorted(exported_check["Ticker"].unique()))
print("Missing values:", int(exported_check.isna().sum().sum()))

print(
    "Duplicate ticker-date rows:",
    int(
        exported_check
        .duplicated(["Ticker", "Date"])
        .sum()
    )
)

print("\nRows per ticker:")
display(
    exported_check["Ticker"]
    .value_counts()
    .sort_index()
)

print("\nRows by split:")
display(
    exported_check["Split"]
    .value_counts()
)

print("\nTrading dates by split:")
display(
    exported_check
    .groupby("Split")["Date"]
    .nunique()
)

display(exported_check.head(8))
display(exported_check.tail(8))

=== EXPORTED FILE CHECK ===
File: data\processed_1y_yahoo\aligned_daily_ohlcv_2025_split.csv
Rows: 1000
Columns: ['Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Quarter', 'Split']
Start date: 2025-01-02
End date: 2025-12-31
Tickers: ['AAPL', 'MSFT', 'NVDA', 'QQQ']
Missing values: 0
Duplicate ticker-date rows: 0

Rows per ticker:


Ticker
AAPL    250
MSFT    250
NVDA    250
QQQ     250
Name: count, dtype: int64


Rows by split:


Split
train         488
validation    256
test          256
Name: count, dtype: int64


Trading dates by split:


Split
test           64
train         122
validation     64
Name: Date, dtype: int64

,Ticker,Date,Open,High,Low,Close,Volume,Quarter,Split
0,AAPL,2025-01-02,247.136526,247.305315,240.077766,242.093140,55740700,Q1,train
1,MSFT,2025-01-02,419.750981,420.283656,409.216031,412.895355,16896500,Q1,train
2,NVDA,2025-01-02,135.797207,138.672917,134.429254,138.103760,198247200,Q1,train
3,QQQ,2025-01-02,510.499434,512.822169,501.972916,506.459534,36389800,Q1,train
4,AAPL,2025-01-03,241.606659,242.420743,240.147249,241.606659,40244100,Q1,train
5,MSFT,2025-01-03,415.361395,418.271344,413.842331,417.600586,16662900,Q1,train
6,NVDA,2025-01-03,139.801207,144.683914,139.521625,144.254562,229322500,Q1,train
7,QQQ,2025-01-03,509.556531,515.810025,508.742644,514.747925,29059500,Q1,train


,Ticker,Date,Open,High,Low,Close,Volume,Quarter,Split
992,AAPL,2025-12-30,272.069428,273.335969,271.540868,272.338684,22139600,Q4,test
993,MSFT,2025-12-30,482.866614,486.592973,482.439332,484.406860,13944500,Q4,test
994,NVDA,2025-12-30,188.010698,188.759785,186.702281,187.311539,97687300,Q4,test
995,QQQ,2025-12-30,618.379832,620.714286,617.761237,617.970764,31226800,Q4,test
996,AAPL,2025-12-31,272.318764,272.937076,271.012322,271.122009,27293600,Q4,test
997,MSFT,2025-12-31,484.764565,485.062692,480.253177,480.571167,15601600,Q4,test
998,NVDA,2025-12-31,189.339064,190.327848,186.262814,186.272797,120100500,Q4,test
999,QQQ,2025-12-31,618.190301,618.499569,612.603457,612.862854,40746500,Q4,test


In [13]:
from pathlib import Path

readme_file = Path("DATA_README.md")

section_title = "## 2025 Quarterly Split Dataset"

split_note = """
## 2025 Quarterly Split Dataset

A separate one-year dataset was created from the cleaned five-year dataset.
The original five-year dataset and the four raw files remain unchanged.

### File

`data/processed_1y_yahoo/aligned_daily_ohlcv_2025_split.csv`

### Dataset details

- Instruments: AAPL, MSFT, NVDA and QQQ
- Date range: 2025-01-02 to 2025-12-31
- Common trading dates per instrument: 250
- Rows per instrument: 250
- Total rows: 1,000
- Missing values: 0
- Duplicate ticker-date rows: 0
- Invalid OHLC rows: 0
- Dates without all four instruments: 0

### Columns

- `Ticker`
- `Date`
- `Open`
- `High`
- `Low`
- `Close`
- `Volume`
- `Quarter`
- `Split`

### Chronological split

- Q1 and Q2: training set
- Q3: validation set
- Q4: test set

The data was not randomly shuffled. The chronological split preserves time
order and helps reduce the risk of future information leakage.

This file contains cleaned OHLCV data and split labels. It does not yet contain
engineered features or the AAPL prediction target.
"""

existing_text = (
    readme_file.read_text(encoding="utf-8")
    if readme_file.exists()
    else ""
)

if section_title not in existing_text:
    updated_text = existing_text.rstrip() + "\n\n" + split_note.strip() + "\n"
    readme_file.write_text(updated_text, encoding="utf-8")
    print("Updated:", readme_file)
else:
    print("The 2025 split section already exists. No duplicate section added.")

print("File exists:", readme_file.exists())
print("Characters:", len(readme_file.read_text(encoding="utf-8")))

Updated: DATA_README.md
File exists: True
Characters: 3500


In [22]:
import shutil
from pathlib import Path

github_processed_dir = Path("data/processed")

github_processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Existing local files
five_year_source = Path(
    "data/processed_5y_yahoo/aligned_daily_ohlcv_5y.csv"
)

one_year_source = Path(
    "data/processed_1y_yahoo/aligned_daily_ohlcv_2025_split.csv"
)

# Standard GitHub destinations
five_year_github = (
    github_processed_dir /
    "aligned_daily_ohlcv_5y.csv"
)

one_year_github = (
    github_processed_dir /
    "aligned_daily_ohlcv_2025_split.csv"
)

# Check source files before copying
print("Five-year source exists:", five_year_source.exists())
print("One-year source exists:", one_year_source.exists())

if not five_year_source.exists():
    raise FileNotFoundError(f"Cannot find: {five_year_source}")

if not one_year_source.exists():
    raise FileNotFoundError(f"Cannot find: {one_year_source}")

# Create GitHub-ready copies
shutil.copy2(
    five_year_source,
    five_year_github
)

shutil.copy2(
    one_year_source,
    one_year_github
)

# Update file paths in DATA_README.md
readme_file = Path("DATA_README.md")
readme_text = readme_file.read_text(encoding="utf-8")

readme_text = readme_text.replace(
    "data/processed_5y_yahoo/aligned_daily_ohlcv_5y.csv",
    "data/processed/aligned_daily_ohlcv_5y.csv"
)

readme_text = readme_text.replace(
    "data/processed_1y_yahoo/aligned_daily_ohlcv_2025_split.csv",
    "data/processed/aligned_daily_ohlcv_2025_split.csv"
)

readme_file.write_text(
    readme_text,
    encoding="utf-8"
)

print("\n=== FILES READY FOR GITHUB ===")
print(five_year_github, five_year_github.exists())
print(one_year_github, one_year_github.exists())
print(readme_file, readme_file.exists())

print("\nFile sizes:")
print(
    "Five-year CSV:",
    five_year_github.stat().st_size,
    "bytes"
)
print(
    "2025 split CSV:",
    one_year_github.stat().st_size,
    "bytes"
)

Five-year source exists: True
One-year source exists: True

=== FILES READY FOR GITHUB ===
data\processed\aligned_daily_ohlcv_5y.csv True
data\processed\aligned_daily_ohlcv_2025_split.csv True
DATA_README.md True

File sizes:
Five-year CSV: 498751 bytes
2025 split CSV: 108914 bytes
